# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [2]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?



**Respuesta:**

- Gymnasium cumple el papel de `Environment`: guarda el mapa, y con `env.step(action)` aplica la acción y devuelve el nuevo estado, la recompensa y si el episodio terminó.
- El agente ya no es una clase; quedó repartido en tres piezas: la **Q-table** (su memoria), `epsilon_greedy_policy` (cómo decide) y `train_q_learning` (cómo aprende).
- Es la misma división de antes, solo que el ambiente ahora es código de librería y el agente es código nuestro.

## 2. Crear FrozenLake


In [3]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?



**Respuesta:**

1. **16 filas**, una por estado: el mapa es 4x4, o sea 16 casillas (`env.observation_space.n`).
2. **4 columnas**, una por acción: Left, Down, Right, Up (`env.action_space.n`).
3. `Q[s,a]` es la **recompensa total que el agente espera obtener** si estando en el estado `s` toma la acción `a` y de ahí en adelante juega lo mejor que sabe. Es un estimado, no un dato exacto: se va corrigiendo con la experiencia.

In [4]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?


**Respuesta:**

- El cero significa **"todavía no sé nada"**, no "todo es malo". Es solo el valor inicial que elegimos; el agente no ha probado ninguna acción.
- Si varias acciones empatan en el valor máximo, `greedy_policy` usa `np.flatnonzero` para juntar todas las empatadas y `np.random.choice` para **elegir una al azar**.
- Sin ese desempate, `np.argmax` devolvería siempre la primera (Left) y al inicio, con toda la tabla en cero, el agente se iría siempre contra la pared izquierda sin conocer el mapa.

## 3. Ejecutar una transición


In [5]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 3
reward     = 0
next_state = 0
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(\quad,\quad,\quad,\quad)
$$

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?


**Respuesta:**

- La experiencia tiene la forma `(s, a, r, s')`. Aquí `s = 0` siempre, porque `env.reset()` deja al agente en la casilla de arriba a la izquierda; `a` salió **al azar** porque pusimos `epsilon = 1.0`; y `r` y `s'` son lo que el ambiente respondió (ejecuta la celda anterior para ver los cuatro números).
- En FrozenLake la recompensa es 0 en casi todos los pasos y 1 solo al llegar a la meta, así que lo normal es que este primer `r` sea 0.0.
- Lo que **no teníamos** en Value Iteration era `r` y `s'` como algo **observado**: allí conocíamos de antemano el modelo `P(s'|s,a)` y `R(s)`, y resolvíamos el problema con álgebra sin ejecutar nunca una acción. Aquí la única forma de saber qué pasa es actuar y mirar.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [6]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [7]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [8]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 0.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [9]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


**Respuesta:**

- Lo único que cambió es la **Q-table**. El mapa, los estados, las acciones y hasta el código que elige la acción (`play_episode` con `random_policy=False`) son idénticos en los tres momentos.
- `Q_initial` está toda en cero, así que el agente no tiene preferencias y camina como si fuera al azar.
- `Q_early` ya tiene algunos valores distintos de cero cerca de la meta, pero la información todavía no se ha propagado a todo el mapa, así que la política puede quedarse a medias.
- `Q_trained` tiene valores en toda la ruta, y al tomar el máximo en cada estado esos números arman el camino completo hasta la meta.

### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** _______

**Valores Q:**

- Left:
- Down:
- Right:
- Up:

¿Cuál acción seleccionaría?:

$$
\arg\max_a Q(s,a)
$$




>


**Respuesta:**

- Para ver la fila de un estado basta con `Q_trained[s]`, que devuelve los cuatro valores en el orden Left, Down, Right, Up.
- La acción que se elegiría es `np.argmax(Q_trained[s])`, que da el **índice** de la acción con mayor valor (0 = Left, 1 = Down, 2 = Right, 3 = Up).
- Interpretación: como la recompensa es 1 solo en la meta y `gamma = 0.95`, el valor de la mejor acción es aproximadamente `0.95` elevado al número de pasos que faltan. Por eso los estados cercanos a la meta tienen valores más altos, y los estados que son hueco quedan en cero: nunca se aprendió nada desde ahí porque el episodio termina.

## 5. Evaluar la política aprendida


In [10]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?


>


**Respuesta:**

- Durante el entrenamiento exploramos para **descubrir** el mapa: sin acciones al azar el agente nunca probaría caminos nuevos y se quedaría con lo primero que le funcionó.
- En la evaluación ya no queremos descubrir nada, queremos **medir qué tan buena quedó la política aprendida**.
- Si exploráramos también al evaluar, una parte de los fracasos sería culpa del azar y no de la política, y el número que reportamos no diría nada útil sobre lo que el agente sabe.

## 6. Experimento: FrozenLake estocástico


In [11]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.772
Std reward : 0.420


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?


**Respuesta:**

- **Ambiente:** con `is_slippery=True` el hielo resbala. La acción que pides se cumple solo una parte de las veces y el resto el agente se desvía a los lados, así que la misma acción desde el mismo estado puede terminar en casillas distintas. Con `is_slippery=False` la acción siempre se cumple.
- **Algoritmo:** la ecuación de Q-Learning **no cambia ni una línea**. Sigue siendo `Q[s,a] += learning_rate * (reward + gamma * max(Q[next_state]) - Q[s,a])`.
- La razón es que Q-Learning nunca usó las probabilidades: el promedio sobre todos los resultados posibles aparece solo por repetir muchas veces la misma transición y actualizar poco a poco.
- El precio es práctico: por eso esta parte del notebook usa 20000 episodios en vez de 10000, y la recompensa media queda bastante por debajo de 1, porque a veces el hielo empuja al agente a un hueco aunque la política sea buena.

# Cierre de clase

Completa antes de terminar:


**Respuestas del cierre:**

**1. ¿Qué almacena Q(s,a)?**

> La recompensa total que el agente espera acumular si desde el estado `s` toma la acción `a` y después sigue actuando lo mejor que sabe. Es un estimado que se va corrigiendo con cada experiencia.

**2. ¿De dónde sale max Q(s',a')?**

> De la **misma Q-table**, leyendo la fila del estado al que acabamos de llegar. Es la estimación actual de "qué tan bueno es el sitio donde caí". Por eso Q-Learning aprende de sus propias estimaciones y no necesita esperar al final del episodio. Cuando el episodio termina se usa 0, porque después del final no hay nada más que ganar.

**3. ¿Por qué necesitamos epsilon-greedy?**

> Porque si el agente siempre eligiera la acción que hoy le parece mejor, nunca probaría las demás y se quedaría atrapado en el primer camino que le funcionó. El `epsilon` lo obliga a probar cosas al azar de vez en cuando, y baja con los episodios para que al final ya explote lo aprendido.

**4. ¿Por qué Q-Learning es model-free?**

> Porque nunca usa `P(s'|s,a)` ni `R(s)`. En Value Iteration esas dos cosas se conocían de antemano y el problema se resolvía con álgebra. Aquí el agente solo ve tuplas `(s, a, r, s')` que salen de actuar en el mundo, y con eso le basta para aprender.